# Demonstration Flamelet Tabulation 101

This notebook builds a premixed-flamelet FLUT from scratch with Cantera, for use with [OxySim-129](https://github.com/STFS-TUDa/oxysim-129), STFS's OpenFOAM framework for carbonaceous solid fuel combustion. See flameletThermo's [README](https://github.com/STFS-TUDa/oxysim-129/blob/main/src/flamelet/flameletThermo/README.md#required-flut-outputs) there for the full, authoritative list of required/optional FLUT outputs (summarized again below).

**Environment:** this notebook needs its own Python environment, per `flut_creation_demo/requirements.txt` (not the same environment as OxySim-129's own test suite):

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

Any other conda/venv/module environment providing those same packages works just as well.

In [ ]:
import cantera as ct 
from matplotlib import pyplot as plt 
import numpy as np
plt.rcParams['figure.dpi'] = 300 #set higher resolution
plt.rcParams['figure.figsize'] = (7, 5)

## 1. Calculating (premixed) flamelets

Load combustion mechanism

In [ ]:
gas = ct.Solution("gri30.yaml")

In [ ]:
Y_fuel = {'CH4': 1}
Y_oxidizer = {'O2': 0.233, 'N2': 0.767}

We start by looking at a single, stoichiometric flame ($\phi = 1$) before running the full parameter variation over $Z$ below.

In [ ]:
gas.TP = 300, ct.one_atm
gas.set_equivalence_ratio(phi=1,fuel=Y_fuel,oxidizer=Y_oxidizer,basis='mass')

Set up the grid refinement criteria (see the [Cantera `set_refine_criteria` docs](https://cantera.org/documentation/docs-3.0/sphinx/html/cython/onedim.html)):
- `ratio`: adds points where the spacing on either side of a grid point differs by more than this factor (default 10; here tightened to 2).
- `slope`: adds points where the value changes by more than this fraction of the total profile range between two adjacent points (default 0.8; here tightened to 0.08).
- `curve`: adds points where the *slope* changes by more than this fraction between two adjacent intervals, i.e. targets curvature (default 0.8; here tightened to 0.15).
- `prune`: removes points where slope/curve are satisfied well below their thresholds; `0.0` disables pruning so the grid never coarsens again.

Tightening `ratio`/`slope`/`curve` below their defaults gives a much finer grid than Cantera's default settings. For this demo the resulting grid has around 170 points, well above the ~100 points we'd want for a well-resolved flame; production tabulations should check point count and refine criteria for their own mechanism/case rather than reusing these values blindly.

In [ ]:
flame = ct.FreeFlame(gas, width=0.1)
flame.set_refine_criteria(ratio=2, slope=0.08, curve=0.15, prune=0.0)

In [ ]:
flame.solve()

In [ ]:
fig, axs = plt.subplots(2,1)
fig.set_size_inches(7,5)
axs[0].plot(flame.grid,flame.T)
axs[0].set_ylabel('T/K')
axs[1].plot(flame.grid,flame.Y[gas.species_index('CO2')],label='CO2')
axs[1].plot(flame.grid,flame.Y[gas.species_index('CO')],label='CO')
axs[1].plot(flame.grid,flame.Y[gas.species_index('H2O')],label='H2O')
axs[1].plot(flame.grid,flame.Y[gas.species_index('H2')]*20,label='H2*20')
axs[1].plot(flame.grid,flame.Y[gas.species_index('OH')]*20,label='OH*20')
axs[1].set_ylabel('Y (-)')
axs[1].set_xlabel('x (m)')
axs[1].legend()
for ax in axs:
    ax.set_xlim(0.032,0.04)
    ax.label_outer()

### Parameter variation over the mixture fraction Z

We now run a series of premixed flames at different mixture fractions $Z$ instead of a single stoichiometric one. Mixture fraction is normally the coordinate used to parameterize *non-premixed* (diffusion) flamelets; here we reuse it to parameterize a family of *premixed* flames at different fuel/oxidizer ratios (`gas.set_mixture_fraction` sets the composition, and each case is still solved as a premixed `FreeFlame`). This gives a 2D manifold spanned by $(Z, y_c)$, similar to the composition space used in FGM/FPI tabulation.

In [ ]:
import tqdm
flames = []
Zs = [0.04,0.05,0.055,0.06,0.07,0.08,0.09,0.1]
for Z in tqdm.tqdm(Zs):
    gas.TP = 300, ct.one_atm
    gas.set_mixture_fraction(Z,fuel=Y_fuel,oxidizer=Y_oxidizer,basis='mass')

    flame = ct.FreeFlame(gas, width=0.1)
    flame.set_refine_criteria(ratio=2, slope=0.08, curve=0.15, prune=0.0)
    flame.solve(loglevel=0)
    flames.append(flame)
print(flames)

In [ ]:
fig, ax = plt.subplots(1,1)
fig.set_size_inches(7,3)
colors = plt.cm.jet(np.linspace(0,1,len(flames)))
for flame, color, Z in zip(flames,colors,Zs):
    ax.plot(flame.grid,flame.T, c=color, label=Z)
ax.set_ylabel('T/K')
ax.set_xlabel('x/m')
ax.legend()
ax.set_xlim(0.032,0.04)

### Put flame data in arrays

Needed data:

Required by OxySim-129's `flameletThermo` for any case:
- `T` (temperature)
- `alpha` (thermal diffusivity)
- `mu` (dynamic viscosity)
- `rho` (density)

Additionally required only if used:
- `W` (mean molecular mass) - only if OxySim-129's radiation model is enabled
- species mass fractions - for `flameletCloudFoam`, one column per species in its Cantera mechanism that isn't itself a transported variable

Everything else built below (`p`, `nu`, `X`) is kept for convenience/demonstration but isn't read by OxySim-129 itself.

In [ ]:
flame_arrays = []
for i, flame in enumerate(flames):
    quantities = []

    # NOTE: the order of quantities.append(...) calls below must match the
    # order of names in the variable_indices dict comprehension at the end
    # of this cell - if you add/reorder one, update the other too.

    # Add Z
    quantities.append(Zs[i] * np.ones(flame.T.shape)[np.newaxis,:])
    # Add temperature -- REQUIRED by OxySim-129's flameletThermo
    quantities.append(flame.T[np.newaxis,:])
    # Add pressure -- not read by OxySim-129 (kept for reference; all flames
    # here are solved at the same constant pressure)
    quantities.append(flame.P * np.ones(flame.T.shape)[np.newaxis,:])
    # Add thermal diffusivity, exported as alpha -- REQUIRED by OxySim-129's
    # flameletThermo
    quantities.append(flame.thermal_conductivity[np.newaxis,:]/flame.density[np.newaxis,:]/flame.cp_mass[np.newaxis,:])
    # Add dynamic viscosity, exported as mu -- REQUIRED by OxySim-129's
    # flameletThermo (it looks up mu by name, not kinematic viscosity)
    quantities.append(flame.viscosity[np.newaxis,:])
    # Add kinematic viscosity (dynamic viscosity / density), exported as nu
    # -- NOT required by OxySim-129; kept only for convenience/demonstration
    quantities.append(flame.viscosity[np.newaxis,:]/flame.density[np.newaxis,:])
    # Add density, exported as rho -- REQUIRED by OxySim-129's flameletThermo
    quantities.append(flame.density[np.newaxis,:])
    # Add mean molecular mass, exported as W -- REQUIRED by OxySim-129's
    # flameletThermo::W() only if the radiation model is enabled
    quantities.append(flame.mean_molecular_weight[np.newaxis,:])
    # Add X (spatial coordinate) -- not read by OxySim-129; only used below
    # to derive yc/omega_yc
    quantities.append(flame.grid[np.newaxis,:])
    # Add species mass fractions -- REQUIRED by OxySim-129's
    # flameletCloudFoam for every species in its Cantera mechanism that
    # isn't itself a transported variable (variablesToSolve)
    quantities.append(flame.Y)

    flame_array = np.concatenate(tuple(quantities))

    flame_arrays.append(flame_array)

variable_indices = {var:index for var,index in zip(["Z","T","p","alpha","mu","nu","rho","W","X"]+gas.species_names,np.arange(0,9+gas.n_species))}

## 2. Challenge: How do we build a manifold from this?

**Map to accessing variables and normalize**

X is not suitable. We need another variable to track the progress of the reaction.

**Requirement:** yc (the progress variable) has to increase monotonically with X and be sensitive in all areas

In [ ]:
fig, ax = plt.subplots(1,1)
fig.set_size_inches(7,3)

for spec in ['CO2', 'CO', 'O2', 'H2O', 'H2', 'CH4']:
    ax.plot(flames[2].grid, flames[2].Y[gas.species_index(spec)],label=spec)
    ax.set_xlim(0.034,0.038)
ax.legend()
ax.set_ylabel('Y (-)')
ax.set_xlabel('x (m)')

In [ ]:
flame = flames[1]

fig, axs = plt.subplots(2,1)
fig.set_size_inches(7,3)

yc_definition = {'CO2': 1, 'CO': 1}

yc = 0
omega_yc = 0
for spec, weight in yc_definition.items():
    yc += flame.Y[gas.species_index(spec)] * weight
    # net_production_rates is molar (kmol/m3/s); multiply by the species'
    # molecular weight to get a mass source term (kg/m3/s).
    omega_yc += flame.net_production_rates[gas.species_index(spec)] * weight * gas.molecular_weights[gas.species_index(spec)]

axs[0].plot(flame.grid, yc,label=spec)

axs[1].plot(flame.grid, omega_yc,label=spec)

for ax in axs:
    ax.set_xlim(0.034,0.04)
    ax.label_outer()
axs[0].set_ylabel('Yc (-)')
axs[1].set_ylabel('$\omega_{yc}$ (kg/m3/s)')
axs[1].set_xlabel('x(m)') 


## Add PV to arrays

In [ ]:
for i, flame in enumerate(flames):
    yc = 0
    omega_yc = 0
    for spec, weight in yc_definition.items():
        yc += flame.Y[gas.species_index(spec)] * weight
        # net_production_rates is molar (kmol/m3/s); multiply by the species'
        # molecular weight to get a mass source term (kg/m3/s).
        omega_yc += flame.net_production_rates[gas.species_index(spec)] * weight * gas.molecular_weights[gas.species_index(spec)]

    flame_arrays[i] = np.concatenate((flame_arrays[i],yc[np.newaxis,:],omega_yc[np.newaxis,:]))
variable_indices.update({"yc":len(variable_indices), "omega_yc":len(variable_indices)+1})

In [ ]:
fig, axs = plt.subplots(2,1)
fig.set_size_inches(7,3)

axs[0].plot(flame_arrays[0][variable_indices["yc"],:], flame_arrays[0][variable_indices["T"],:])
axs[0].plot(flame_arrays[2][variable_indices["yc"],:], flame_arrays[2][variable_indices["T"],:])
axs[0].plot(flame_arrays[7][variable_indices["yc"],:], flame_arrays[7][variable_indices["T"],:])

axs[1].plot(flame_arrays[0][variable_indices["yc"],:], flame_arrays[0][variable_indices["omega_yc"],:])
axs[1].plot(flame_arrays[2][variable_indices["yc"],:], flame_arrays[2][variable_indices["omega_yc"],:])
axs[1].plot(flame_arrays[7][variable_indices["yc"],:], flame_arrays[7][variable_indices["omega_yc"],:])

axs[0].set_ylabel('T (K)')
axs[1].set_ylabel('$\omega_{yc}$ (kg/m3/s)')
axs[1].set_xlabel('Yc (-)') 
for ax in axs:
    ax.label_outer()

## 3. Assemble database

**Build a cube**

The FLUT is a single dense array, so every flamelet needs to end up with the same number of grid points, sitting on the same grid along $y_c$. Right now they don't: each flame has its own $y_c$ range and its own number of grid points (Cantera's adaptive refinement stops at a different point count per flame). We fix this in two steps: (1) normalize each flame's $y_c$ to a common $[0,1]$ range below, then (2) interpolate every flame onto the same $y_{c,norm}$ grid further down.

We want to map everything to normalized yc (normalized to max and min for a given Z).

yc_norm = $ \frac{yc - yc_{min}}{yc_{max} - yc_{min}}$

In [ ]:
for i,_ in enumerate(flame_arrays):
    yc = flame_arrays[i][variable_indices["yc"]]

    # yc_min is fixed to 0 rather than np.min(yc): yc is assumed to be zero
    # at the unburnt inlet for every flame, so all flames share the same
    # (physically meaningful) lower bound instead of a numerically-found one.
    yc_min, yc_max = 0, np.max(yc)
    yc_norm = (yc - yc_min) / (yc_max - yc_min + 1e-16)
    yc_norm[0], yc_norm[-1] = 0, 1

    flame_arrays[i] = np.concatenate((flame_arrays[i], yc_norm[np.newaxis,:], yc_min * np.ones(yc.shape)[np.newaxis,:], yc_max * np.ones(yc.shape)[np.newaxis,:]))
variable_indices.update({var:index for var,index in zip(["yc_norm","yc_min","yc_max"],np.arange(len(variable_indices),len(variable_indices)+3+1))})

In [ ]:
fig, axs = plt.subplots(2,1)
fig.set_size_inches(7,3)

axs[0].plot(flame_arrays[0][variable_indices["yc_norm"],:], flame_arrays[0][variable_indices["T"],:])
axs[0].plot(flame_arrays[2][variable_indices["yc_norm"],:], flame_arrays[2][variable_indices["T"],:])
axs[0].plot(flame_arrays[7][variable_indices["yc_norm"],:], flame_arrays[7][variable_indices["T"],:])

axs[1].plot(flame_arrays[0][variable_indices["yc_norm"],:], flame_arrays[0][variable_indices["omega_yc"],:])
axs[1].plot(flame_arrays[2][variable_indices["yc_norm"],:], flame_arrays[2][variable_indices["omega_yc"],:])
axs[1].plot(flame_arrays[7][variable_indices["yc_norm"],:], flame_arrays[7][variable_indices["omega_yc"],:])

axs[0].set_ylabel('T (K)')
axs[1].set_ylabel('$\omega_{yc}$ (kg/m3/s)')
axs[1].set_xlabel('Yc (-)') 
for ax in axs:
    ax.label_outer()

### Interpolate to common grid

Each flame has a different number of grid points (Cantera's adaptive refinement stops at a different point count for each one, as printed below), and even after normalizing to $y_{c,norm} \in [0,1]$ the flames don't sit on the same $y_{c,norm}$ values. To stack the flames into a single array we need every flame evaluated at the *same* set of $y_{c,norm}$ points, so we interpolate each flame's data onto a common, regularly spaced `yc_norm_grid`.

In [ ]:
print([flame_array.shape for flame_array in flame_arrays])

In [ ]:
yc_norm_grid = np.linspace(0,1,101)

from scipy import interpolate 

for i,_ in enumerate(flame_arrays):
    # flame_arrays[i] has shape (n_vars, n_points); interp1d's default axis=-1
    # interpolates every row (every variable) independently against the same
    # x-values (yc_norm), so one call re-samples all variables at once onto
    # yc_norm_grid.
    interpolation = interpolate.interp1d(flame_arrays[i][variable_indices["yc_norm"]], flame_arrays[i])
    flame_arrays[i] = interpolation(yc_norm_grid)

In [ ]:
print([flame_array.shape for flame_array in flame_arrays])

### Rearrange arrays and concatenate

Each `flame_arrays[i]` is currently shaped `(n_vars, n_yc_norm_grid)`. We want the final table (`flut`) to have the convention `(n_Z, n_yc_norm_grid, n_vars)` - one axis per input dimension ($Z$, then $y_{c,norm}$), with the variables along the last axis. So each flame's array is transposed to `(n_yc_norm_grid, n_vars)`, given a new leading axis with `np.newaxis`, and all flames are concatenated along that new axis (one entry per $Z$).

In [ ]:
flame_arrays_rearranged = []
for i,_ in enumerate(flame_arrays):
    flame_arrays_rearranged.append(flame_arrays[i].T[np.newaxis,...])

In [ ]:
flut=np.concatenate(tuple(flame_arrays_rearranged))

In [ ]:
flut.shape

### Plot manifold

In [ ]:
fig, axs = plt.subplots(1,3)
fig.set_size_inches(7,3)

for ax,var in zip(axs,["T","omega_yc","CO2"]):
    cs = ax.tricontourf(flut[:,:,variable_indices["Z"]].ravel(),flut[:,:,variable_indices["yc"]].ravel(),flut[:,:,variable_indices[var]].ravel())
    fig.colorbar(cs, ax=ax)
    # ax.set_ylabel("yc/-")
    ax.label_outer()
    ax.set_title(var)
axs[1].set_xlabel("Z (-)")

In [ ]:
fig, axs = plt.subplots(1,3)
fig.set_size_inches(7,3)

for ax,var in zip(axs,["T","omega_yc","CO2"]):
    cs = ax.tricontourf(flut[:,:,variable_indices["Z"]].ravel(),flut[:,:,variable_indices["yc_norm"]].ravel(),flut[:,:,variable_indices[var]].ravel())
    fig.colorbar(cs, ax=ax)
    # ax.set_ylabel("yc/-")
    ax.label_outer()
    ax.set_title(var)
axs[1].set_xlabel("Z (-)")
axs[0].set_ylabel("yc_norm (-)")

In [ ]:
import h5py

from h5_utilities import write_dictionary_to_hdf5

file_name = "manual_flut.h5"
compression = None

# Input variables (independent axes of the flut cube, in axis order 0, 1, ...)
input_dict = {
    "Z": np.array(Zs),
    "yc_norm": yc_norm_grid,
}

# Output variables: everything in variable_indices that is not an input axis,
# kept in their original column order and reindexed to 0..n_outputs-1
output_names = [
    name
    for name, _ in sorted(variable_indices.items(), key=lambda item: item[1])
    if name not in input_dict
]
output_dict = {name: i for i, name in enumerate(output_names)}
data = flut[..., [variable_indices[name] for name in output_names]]

metadata = {
    "mechanism": "gri30.yaml",
    "fuel": Y_fuel,
    "oxidizer": Y_oxidizer,
    "yc_definition": yc_definition,
}

# yc_norm is (yc - yc_min) / (yc_max - yc_min); yc_min/yc_max are stored as
# output fields, so the reader can reconstruct the un-normalized yc.
normalization_dict = {
    "Z": {"Normalize": False},
    "yc_norm": {
        "Normalize": True,
        "Settings": {
            "Normalization_type": "default",
            "Physical_name": "yc",
            "Min_name_normalization": "yc_min",
            "Max_name_normalization": "yc_max",
        },
    },
}

pyflut_settings = {
    "key_variable": "yc_norm",
    "fuel": Y_fuel,
    "oxidizer": Y_oxidizer,
}

with h5py.File(file_name, "w") as h5:
    # input_dict
    for i, (key, value) in enumerate(input_dict.items()):
        write_dictionary_to_hdf5(
            h5.require_group(f"input_dict/{key}"),
            {"Values": value, "Index": int(i)},
        )

    # output_dict
    write_dictionary_to_hdf5(h5, {"output_dict": output_dict})

    # data
    h5.create_dataset("data", data=data, compression=compression)

    # metadata
    write_dictionary_to_hdf5(h5, {"metadata": metadata}, encode_list_as_string=True)

    # Normalization settings (used by the OpenFOAM flut-reader)
    for key in input_dict.keys():
        write_dictionary_to_hdf5(
            h5.require_group(f"input_dict/{key}"),
            {"Normalization_settings": normalization_dict[key]},
        )

    # pyflut-internal settings
    write_dictionary_to_hdf5(h5.require_group("pyflut_settings"), pyflut_settings)

print(f"Wrote flut to {file_name}")
